In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

# EDA

In [ ]:
df.head()

In [ ]:
print(f"{'Features':<20}{'unique_vals':<15}{'missing_vals':<15}{'dtype'}\n")
for i in df.columns:
    unique_vals = len(df[i].unique())
    missing_vals = df[i].isna().sum()
    dtype = df[i].dtype


    print(f'{i:<20}{unique_vals:<15}{missing_vals:<15}{dtype}')
print(f'\nshape : {df.shape}')

In [ ]:
catagorical_cols = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked']
target_col = 'Survived'

for i in catagorical_cols:
    
    sns.countplot(x = i, data = df, order = set(df[i]), hue = target_col)
    plt.title(f'{i} distribution')
    plt.show()

In [ ]:
import warnings
warnings.simplefilter("ignore", category=FutureWarning)

g = sns.displot(data = df[df['Age'].notna()], x = 'Age', hue = 'Sex', col = target_col, multiple = 'dodge', shrink = 0.7)

g.fig.suptitle('Age and Sex correlation with survival')
plt.subplots_adjust(top=0.85)

plt.show()


# Pipeline

## 1. preprocessing

In [ ]:
class Impute(BaseEstimator, TransformerMixin):

    def fit(self, X , y=None):
        X = X.copy()
        self.Embarked_missval_ = X['Embarked'].mode()[0]
        self.cabin_missval_ = 'Unknown'
        self.Age_missval_ = X.groupby('Pclass')['Age'].median()
        self.fare_mode_ = X['Fare'].mode()[0]
        return self

    def transform(self, X,  y=None):
        X = X.copy()
        X.fillna({'Embarked': self.Embarked_missval_}, inplace = True)
        X.fillna({'Cabin': self.cabin_missval_}, inplace = True)
        X['Age'] = X['Age'].fillna(X['Pclass'].map(self.Age_missval_))
        X.fillna({'Fare': self.fare_mode_}, inplace = True)
        return X

## 2. Feature Engineering / Encoding

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoder = OneHotEncoder(handle_unknown = 'ignore')
    
    def fit(self, X,  y=None):
        X = X.copy()
        self.encoder.fit(X[['Embarked']])

        self.fare_bins_ = np.unique(np.quantile(X['Fare'].fillna(X['Fare'].median()), [0.0, 0.25, 0.5, 0.75, 1.0]))

        if len(self.fare_bins_) < 5:
            # fall back to simple equal-width bins
            self.fare_bins_ = np.linspace(X['Fare'].min(), X['Fare'].max(), 5)
        return self


    def transform(self, X,  y=None):
        X = X.copy()
        
        X['Title'] = X['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
        X['Title'] = X['Title'].replace(
            ['Mlle', 'Ms'], 'Miss'
        ).replace(['Mme'], 'Mrs').replace(
            ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 
             'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
        )
        title_map = {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Rare':4}
        X['Title'] = X['Title'].map(title_map).fillna(4).astype(int)

        X['CabinDeck'] = X['Cabin'].astype(str).str[0]
        deck_map = {'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7,'T':8}
        X['CabinDeck'] = X['CabinDeck'].map(deck_map).fillna(0).astype(int)

        
        
        X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
        X['hasCabin'] = (X['Cabin'] != 'Unknown').astype(int)
        X['Age_bin'] = pd.cut(X['Age'] , bins = [0,12,18,35,50,80], labels=[0,1,2,3,4], include_lowest = True).astype(int)
        X['Fare_bin'] = pd.cut(X['Fare'].fillna(self.fare_bins_.mean()), bins=self.fare_bins_, labels=[0,1,2,3], include_lowest=True).astype(int)
        X['Family_size'] = X['SibSp'] + X['Parch'] + 1
        X['isAlone'] = (X['Family_size'] == 1).astype(int)

        ticket_counts = X['Ticket'].value_counts()
        X['TicketGroupSize'] = X['Ticket'].map(ticket_counts)
        X['TicketGroupSize'] = pd.cut(X['TicketGroupSize'],
                                      bins=[0,1,3,10],
                                      labels=[0,1,2]).astype(int)
        X['FamilySurvival'] = np.where(X['Family_size'] > 1, 1, 0)
        
        embarked_encoded = self.encoder.transform(X[['Embarked']])
        embarked_encoded = pd.DataFrame(
            embarked_encoded.toarray().astype(int),
            columns=self.encoder.get_feature_names_out(['Embarked']),
            index=X.index
        )

        X['Sex*Pclass'] = X['Sex'] * X['Pclass']
        
        X = pd.concat([X.drop(columns=['Embarked']), embarked_encoded], axis=1)
        X.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis = 1, inplace = True)

        X.fillna(0, inplace=True)
        
        return X

## 3. Model

In [ ]:
xgb_model = XGBClassifier(random_state=42)
rf_model  = RandomForestClassifier(random_state=42)
lr_model  = LogisticRegression(max_iter = 1000 , random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('rf', rf_model),
        ('lr', lr_model)
    ],
    voting='soft'
)


pipe = Pipeline([
    ('impute', Impute()),
    ('feature_engineering', FeatureEngineering()),
    ('model', ensemble)
])

In [ ]:
X = df.drop(columns = ['Survived'])
y = df['Survived']

## 4. hyperparameter tuning / Cross-Validation

In [ ]:
param_dict = {
    'model__xgb__n_estimators': [200, 400, 800],
    'model__xgb__learning_rate': [0.01, 0.05, 0.1],
    'model__xgb__max_depth': [5, 6, 7, 8],
    'model__xgb__min_child_weight': [4, 5, 6],
    'model__xgb__subsample': [0.7, 0.8, 1.0],
    'model__xgb__colsample_bytree': [0.7, 0.8, 1.0],
    'model__xgb__gamma': [0, 0.1, 0.3],
    'model__xgb__reg_lambda': [1, 1.5, 2],

    'model__rf__n_estimators': [300, 500, 700],
    'model__rf__max_depth': [6, 7, 8],
    'model__rf__min_samples_split': [2, 4, 6],

    'model__lr__C': [0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = RandomizedSearchCV(
    pipe,
    param_distributions = param_dict,
    n_iter=25,
    scoring='accuracy',
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state = 42
)

grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

# Model Execution

In [ ]:
best_pipe = grid.best_estimator_

In [ ]:
from sklearn.model_selection import cross_val_predict
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = cross_val_predict(best_pipe, X, y, cv=cv, method='predict')
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("OOF accuracy:", accuracy_score(y, oof_preds))
print(classification_report(y, oof_preds))

In [ ]:
predictions = best_pipe.predict(test)
res = pd.DataFrame()
res['PassengerId'] = test['PassengerId']
res['Survived'] = predictions
res.to_csv('submission.csv', index=False)